<a id="vlm-sandbox-models"></a>
# VideoDB Understanding: VLM with Sandbox Models

Run Qwen or Gemma vision-language models on a compatible VideoDB sandbox.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/vlm/sandbox-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install and connect

Sandbox Compute is useful when you want a private/self-hosted model rather than a managed provider model. Sandboxes are billable.

In [ ]:
!pip install -q videodb python-dotenv

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import SandboxModel, SandboxTier, connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()

print("Connected to VideoDB")
print(f"Collection: {collection.id}")

## 2. Choose a video

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"
video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# video = collection.get_video("m-...")

print("Video:", video.id)
video.play()

<a id="models"></a>
## 3. Choose a sandbox VLM

| Tier | Example vision models |
|---|---|
| Small | `Qwen/Qwen3.5-9B`, `google/gemma-4-E2B-it` |
| Medium | `Qwen/Qwen3.5-27B`, `google/gemma-4-26B-A4B-it`, `google/gemma-4-31B-it` |

The requested model must be compatible with the sandbox tier.

In [ ]:
SANDBOX_TIER = SandboxTier.small
VLM_MODEL = SandboxModel.QWEN_9B

print("Sandbox configuration:")
print(f"- Tier: {SANDBOX_TIER}")
print(f"- Model: {VLM_MODEL.value}")

## 4. Create the sandbox

In [ ]:
sandbox = conn.create_sandbox(tier=SANDBOX_TIER)
print("Sandbox created")
print(f"ID: {sandbox.id}")
print(f"Status: {sandbox.status}")

### Wait until the sandbox is active

Run this cell again if the sandbox is still provisioning. It reuses the sandbox created above instead of creating another one.


In [ ]:
sandbox.wait_for_ready(timeout=300, interval=5)
print("Sandbox ready")
print(f"ID: {sandbox.id}")
print(f"Status: {sandbox.status}")

## 5. Run the VLM

Supplying both the sandbox model and `sandbox_id` selects the authorized Open Compute destination.

In [ ]:
understanding = video.understand(
    analyzers=[{
        "type": "vlm",
        "name": "scene",
        "sampling": {"strategy": "uniform", "frame_count": 6},
        "config": {
            "model": VLM_MODEL,
            "sandbox_id": sandbox.id,
            "prompt": "Describe the people, objects, actions, and setting in this scene.",
        },
    }],
    segmentation={"type": "shot", "threshold": 30},
)

understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Understanding complete")
print(f"ID: {understanding.id}")
print(f"Status: {understanding.status}")

## 6. Inspect output

In [ ]:
from pprint import pprint

scene_output = understanding.get_analyzer("scene").get_output()
scenes = scene_output.get("scenes", [])

print(f"Previewing {min(len(scenes), 5)} of {len(scenes)} scenes")
print("-" * 60)
for scene in scenes[:5]:
    print(f"\n{scene.get('start')}s → {scene.get('end')}s")
    pprint(scene.get("data") or {}, width=100, sort_dicts=False)

## Common failures

- **No active compatible sandbox:** create the correct tier and wait for `active`.
- **Incompatible tier:** select a model listed for the sandbox tier.
- **Unknown model:** use a model exposed by the Sandbox model catalogue.
- **Stopped sandbox:** provision or activate a usable sandbox before submitting.

Understanding never provisions a billable sandbox implicitly.

## Related guides

- [VLM with Managed Models](managed-models.ipynb)
- [Understanding guide map](../README.md)


## 7. Cleanup

In [ ]:
DELETE_RUN = False
STOP_SANDBOX = True

if DELETE_RUN:
    understanding.delete()
if STOP_SANDBOX:
    sandbox.stop()
    print("Stopping", sandbox.id)